In [0]:
%sql
-- 16 Tim Hortons Roll Up to Win 2025 gift codes
CREATE OR REPLACE TEMP VIEW promo_codes AS
SELECT code FROM (
  VALUES
    ('GIFT12C-KQHQ-4FT4-BMRH-MP3Q'),
    ('GIFT12C-MFUU-E2D6-AVBP-D7VK'),
    ('GIFT12C-8PVD-KYVM-MCWK-LW1P'),
    ('GIFT12C-UL42-X45F-4PGX-P38X'),
    ('GIFT12C-XTVE-N0G7-V6K4-1109'),
    ('GIFT12C-KQ3A-E86L-V8C4-JKKV'),
    ('GIFT12C-Y93U-XNKX-PD8T-QURU'),
    ('GIFT12C-872Q-C5N1-CCNJ-BBVQ'),
    ('GIFT12C-LHTD-Q43H-QH9X-URME'),
    ('GIFT12C-JR5B-A7Q1-DC3K-FUCL'),
    ('GIFT12C-XDVF-88YJ-7UNH-BXEX'),
    ('GIFT12C-B5YD-3F1M-YDUK-BDEH'),
    ('GIFT12C-N5N7-YFB2-X1C4-71V2'),
    ('GIFT12C-T23B-KNDM-AUJB-TUKX'),
    ('GIFT12C-G89Q-C2WD-EXBH-YYCD'),
    ('GIFT12C-6J7C-QWYQ-NMWQ-9UV9')
) t(code);

-- Resolve gift codes to redeemer identifiers
CREATE OR REPLACE TEMP VIEW tims_redeemers AS
SELECT
  p.code                            AS gift_code,
  g.redeemer_id,                    -- bigint; maps to cc_user_profile_id
  g.redeemer_plan_id,
  g.redemption_date,
  g.is_redeemed
FROM promo_codes p
LEFT JOIN cpx_dataanalytics_silver.cineclub.gifting g
  ON g.id = p.code;

SELECT * FROM tims_redeemers ORDER BY is_redeemed DESC, redemption_date;

In [0]:
%sql
-- Join redeemers to the latest profile snapshot to get CDE_ID + current membership status
-- Latest available snapshot: 20260714
CREATE OR REPLACE TEMP VIEW tims_member_status AS
SELECT
  r.gift_code,
  r.redeemer_id,
  r.redemption_date,
  r.redeemer_plan_id,
  s.cde_id,
  s.plan_id                        AS current_plan_id,
  s.renewal_status,
  s.renewal_datetime,
  s.activated_at_datetime,
  s.current_term_start_datetime,
  s.current_term_end_datetime,
  s.annual_current_term_end_datetime
FROM tims_redeemers r
LEFT JOIN cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot s
  ON s.cc_user_profile_id = r.redeemer_id
  AND s.snapshot_date_id = '20260714';

SELECT *
FROM tims_member_status
ORDER BY redemption_date;

In [0]:
%sql
-- Consolidate member windows once, then scan the transaction fact only once for the full cohort
CREATE OR REPLACE TEMP VIEW tims_members AS
SELECT
  gift_code,
  CAST(cde_id AS INT)                  AS cde_id,
  redemption_date,
  redemption_date                      AS gift_start,
  redemption_date + INTERVAL 1 YEAR    AS gift_end,
  redemption_date - INTERVAL 1 YEAR    AS pre_start,
  redemption_date                      AS pre_end,
  renewal_status,
  current_plan_id
FROM tims_member_status
WHERE cde_id IS NOT NULL;

CREATE OR REPLACE TEMP VIEW tims_member_metrics AS
WITH bounds AS (
  SELECT
    MIN(pre_start)  AS min_txn_ts,
    MAX(gift_end)   AS max_txn_ts
  FROM tims_members
),
cohort_txns AS (
  SELECT
    /*+ BROADCAST(m) */
    m.gift_code,
    m.cde_id,
    m.renewal_status,
    m.gift_start,
    m.gift_end,
    m.pre_end,
    t.CDE_Visit_ID,
    t.Transaction_DateTime,
    CASE WHEN t.TicketFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END      AS box_office_rev,
    CASE WHEN t.ConcessionFLAG = 'Yes' THEN t.Net_Revenue ELSE 0 END  AS concessions_rev,
    t.Net_Revenue                                                      AS total_rev,
    -- Visit = unique (CDE_ID, VisitDateId, LocationId) tuple; CDE_Visit_ID is often NULL
    CONCAT_WS('|', CAST(t.CDE_ID AS STRING), CAST(t.VisitDateId AS STRING), CAST(t.LocationId AS STRING)) AS visit_key
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  CROSS JOIN bounds b
  JOIN tims_members m
    ON  t.CDE_ID               = m.cde_id
    AND t.Transaction_DateTime >= m.pre_start
    AND t.Transaction_DateTime  < m.gift_end
  WHERE t.Transaction_DateTime >= b.min_txn_ts
    AND t.Transaction_DateTime  < b.max_txn_ts
    AND (t.TicketFLAG = 'Yes' OR t.ConcessionFLAG = 'Yes')
)
SELECT
  m.gift_code,
  m.cde_id,
  m.renewal_status,
  DATE(m.gift_start)                                                                AS gift_year_start,
  DATE(m.gift_end)                                                                  AS gift_year_end,
  COUNT(DISTINCT CASE WHEN x.Transaction_DateTime <  m.pre_end    THEN x.visit_key END) AS pre_visits,
  ROUND(SUM(CASE WHEN x.Transaction_DateTime <  m.pre_end    THEN x.box_office_rev   ELSE 0 END), 2) AS pre_box_office,
  ROUND(SUM(CASE WHEN x.Transaction_DateTime <  m.pre_end    THEN x.concessions_rev  ELSE 0 END), 2) AS pre_concessions,
  ROUND(SUM(CASE WHEN x.Transaction_DateTime <  m.pre_end    THEN x.total_rev        ELSE 0 END), 2) AS pre_total,
  COUNT(DISTINCT CASE WHEN x.Transaction_DateTime >= m.gift_start THEN x.visit_key END) AS post_visits,
  ROUND(SUM(CASE WHEN x.Transaction_DateTime >= m.gift_start THEN x.box_office_rev   ELSE 0 END), 2) AS post_box_office,
  ROUND(SUM(CASE WHEN x.Transaction_DateTime >= m.gift_start THEN x.concessions_rev  ELSE 0 END), 2) AS post_concessions,
  ROUND(SUM(CASE WHEN x.Transaction_DateTime >= m.gift_start THEN x.total_rev        ELSE 0 END), 2) AS post_total
FROM tims_members m
LEFT JOIN cohort_txns x
  ON  m.gift_code = x.gift_code
  AND m.cde_id    = x.cde_id
GROUP BY
  m.gift_code,
  m.cde_id,
  m.renewal_status,
  DATE(m.gift_start),
  DATE(m.gift_end),
  m.pre_end,
  m.gift_start;

SELECT
  gift_code,
  cde_id,
  renewal_status,
  gift_year_start,
  gift_year_end,
  post_visits                              AS visits,
  post_box_office                          AS box_office_net_rev,
  post_concessions                         AS concessions_net_rev,
  post_total                               AS total_net_rev
FROM tims_member_metrics
ORDER BY total_net_rev DESC, gift_code;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tims_prepost AS
SELECT * FROM tims_member_metrics;

-- ── Per-member pre / post with deltas ─────────────────────────────────────────
SELECT
  gift_code,
  cde_id,
  renewal_status,
  pre_visits,
  post_visits,
  post_visits - pre_visits                        AS visit_delta,
  pre_box_office,
  post_box_office,
  ROUND(post_box_office  - pre_box_office,  2)   AS box_office_delta,
  pre_concessions,
  post_concessions,
  ROUND(post_concessions - pre_concessions, 2)   AS concessions_delta,
  pre_total,
  post_total,
  ROUND(post_total       - pre_total,       2)   AS total_delta
FROM tims_prepost
ORDER BY total_delta DESC, gift_code;

In [0]:
%sql
-- Cohort-level totals: incremental box office + concessions vs prior year
SELECT
  COUNT(*)                                               AS cohort_size,
  SUM(pre_visits)                                        AS total_pre_visits,
  SUM(post_visits)                                       AS total_gift_yr_visits,
  SUM(post_visits) - SUM(pre_visits)                     AS incremental_visits,
  ROUND(SUM(pre_box_office),  2)                         AS total_pre_box_office,
  ROUND(SUM(post_box_office), 2)                         AS total_gift_yr_box_office,
  ROUND(SUM(post_box_office)  - SUM(pre_box_office), 2) AS incremental_box_office,
  ROUND(SUM(pre_concessions),  2)                        AS total_pre_concessions,
  ROUND(SUM(post_concessions), 2)                        AS total_gift_yr_concessions,
  ROUND(SUM(post_concessions) - SUM(pre_concessions), 2) AS incremental_concessions,
  ROUND(SUM(pre_total),  2)                              AS total_pre_revenue,
  ROUND(SUM(post_total), 2)                              AS total_gift_yr_revenue,
  ROUND(SUM(post_total)  - SUM(pre_total), 2)           AS incremental_total_revenue
FROM tims_member_metrics;